# Relation KG Translation Experiment (Draft)

이 노트북은 `관계 그래프 기반 번역 컨텍스트`를 빠르게 실험하기 위한 초안입니다.

실험 흐름:
1. 관계 후보 추출
2. 샘플별 관계 컨텍스트 생성
3. D 조건 번역 실행 (옵션)
4. A/B/C/D 비교 시트 생성


In [ ]:
# Optional installs
# %pip install -q pandas openai tqdm


In [ ]:
import os
import subprocess
from pathlib import Path
import pandas as pd

BASE = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
SCRIPTS = BASE / 'scripts'
DATA = BASE / 'data'
REL = DATA / 'relation_kg'
OUT = BASE / 'outputs'

print('BASE =', BASE)


In [ ]:
# 1) Extract relation candidates + auto edges
cmd = [
    'python3', str(SCRIPTS / 'extract_relation_candidates.py'),
    '--allow-cooccurrence'
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

cand = pd.read_csv(REL / 'relation_candidates.csv')
auto = pd.read_csv(REL / 'relation_edges_auto.csv')
print('candidates:', len(cand), 'auto edges:', len(auto))
print(cand['relation'].value_counts().head(10))


In [ ]:
# 2) Build sample-level relation context
cmd = [
    'python3', str(SCRIPTS / 'build_relation_context.py'),
    '--samples-csv', str(DATA / 'samples_tagged_v1.csv'),
    '--edges-csv', str(REL / 'relation_edges_auto.csv'),
    '--out-csv', str(REL / 'sample_relation_context.csv')
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

ctx = pd.read_csv(REL / 'sample_relation_context.csv')
print('context rows:', len(ctx))
ctx.head(10)


In [ ]:
# 3) Quick inspection: rows where context found
ctx = pd.read_csv(REL / 'sample_relation_context.csv')
has_ctx = ctx[~ctx['relation_context'].str.contains('No explicit relation context', na=False)]
print('with context:', len(has_ctx), '/', len(ctx))
has_ctx[['sample_id','sentence_type','source_text','detected_entities','relation_context']].head(20)


In [ ]:
# 4) Run D condition translation
# Set OPENAI_API_KEY in environment first if dry_run=False
DRY_RUN = True
RUN_DATE = '2026-04-12'

cmd = [
    'python3', str(SCRIPTS / 'run_condition_d.py'),
    '--run-date', RUN_DATE,
]
if DRY_RUN:
    cmd.append('--dry-run')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

d_path = OUT / f'run_{RUN_DATE}' / 'D_outputs.csv'
df_d = pd.read_csv(d_path)
print('D outputs:', len(df_d))
df_d.head(10)


In [ ]:
# 5) Build A/B/C/D comparison sheet
RUN_DATE = '2026-04-12'
run_dir = OUT / f'run_{RUN_DATE}'

samples = pd.read_csv(DATA / 'samples.csv')
A = pd.read_csv(run_dir / 'A_outputs.csv') if (run_dir / 'A_outputs.csv').exists() else None
B = pd.read_csv(run_dir / 'B_outputs.csv') if (run_dir / 'B_outputs.csv').exists() else None
C = pd.read_csv(run_dir / 'C_outputs.csv') if (run_dir / 'C_outputs.csv').exists() else None
D = pd.read_csv(run_dir / 'D_outputs.csv') if (run_dir / 'D_outputs.csv').exists() else None

merged = samples[['sample_id','sentence_type','source_text']].copy()
if A is not None:
    merged = merged.merge(A[['sample_id','translation']].rename(columns={'translation':'condition_A'}), on='sample_id', how='left')
if B is not None:
    merged = merged.merge(B[['sample_id','translation']].rename(columns={'translation':'condition_B'}), on='sample_id', how='left')
if C is not None:
    merged = merged.merge(C[['sample_id','translation']].rename(columns={'translation':'condition_C'}), on='sample_id', how='left')
if D is not None:
    merged = merged.merge(D[['sample_id','translation']].rename(columns={'translation':'condition_D'}), on='sample_id', how='left')

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'D_meaning','D_term','D_consistency','D_naturalness','D_ui_fit',
    'best','error_type','notes'
]:
    if col not in merged.columns:
        merged[col] = ''

out_eval = BASE / 'eval' / f'eval_sheet_prefilled_ABCD_{RUN_DATE}.csv'
merged.to_csv(out_eval, index=False, encoding='utf-8')
print('saved:', out_eval)
print('rows:', len(merged))


## Notes
- `relation_edges_auto.csv`는 자동 규칙 결과이므로 잡음이 있습니다.
- 발표/보고용 실험에서는 `relation_edges_confirmed.csv`를 만들어 수동 확정본으로 재실행하는 것을 권장합니다.